![Finance Toolkit](https://github.com/JerBouma/FinanceToolkit/assets/46355364/198d47bd-e1b3-492d-acc4-5d9f02d1d009)

**The FinanceToolkit** is an open-source toolkit in which all relevant financial methods (500+) are written down in the most simplistic way allowing for complete transparency of the calculation method. This allows you to not have to rely on metrics from other providers and, given a financial statement, allow for efficient manual calculations. This leads to one uniform method of calculation being applied that is available and understood by everyone.

# Installation
To install the FinanceToolkit it simply requires the following:

```
pip install financetoolkit -U
```

From within Python use:

```python
from financetoolkit import Toolkit
```
 
To be able to get started, you need to obtain an API Key from FinancialModelingPrep. This is used to gain access to 30+ years of financial statement both annually and quarterly. Note that the Free plan is limited to 250 requests each day, 5 years of data and only features companies listed on US exchanges.

___ 

<b><div align="center">Obtain an API Key from FinancialModelingPrep <a href="https://www.jeroenbouma.com/fmp" target="_blank">here</a>.</div></b>
___

Through the link you are able to subscribe for the free plan and also premium plans at a **15% discount**. This is an affiliate link and thus supports the project at the same time. I have chosen FinancialModelingPrep as a source as I find it to be the most transparent, reliable and at an affordable price. When you notice that data is inaccurate or have any other issue related to the data, note that I simply provide the means to access this data and I am not responsible for the accuracy of the data itself. For this, use <a href="https://site.financialmodelingprep.com/contact" target="_blank">their contact form</a> or provide the data yourself.

In [ ]:
import pandas as pd

from financetoolkit import Toolkit

API_KEY = "FINANCIAL_MODELING_PREP_API_KEY"

The Econometrics module provides a broad set of regression, hypothesis-testing, time-series and panel-data methods -- Ordinary Least Squares (OLS) and other regression estimators, unit root and cointegration tests, Granger causality, causal inference (IV/DiD/RDD/PSM/Synthetic Control), panel data (Fixed/Random Effects) and time-series forecasting (ARIMA, VAR, VECM). It is accessed through `toolkit.econometrics` and requires the optional `financetoolkit[econometrics]` extra (`pip install financetoolkit[econometrics]`), which pulls in `statsmodels` and `linearmodels`.

This notebook follows one investigation start to finish: **is Apple's stock actually tied to its chip suppliers and megacap peers, or just eyeballed pairwise correlation?** Tickers include Apple's RF/modem/foundry suppliers (`QCOM`, `SWKS`, `TSM`), other megacap tech (`MSFT`, `GOOGL`, `AMZN`, `META`, `NVDA`), and two unrelated-sector names as a contrast (`XOM`, `PG`) -- the Benchmark is deliberately omitted so results aren't just "the whole market moving together".

In [2]:
# A deliberately wide, varied universe: Apple's suppliers, megacap tech peers and
# two unrelated names (XOM, PG) for contrast. No Benchmark.
companies = Toolkit(
    ["AAPL", "TSM", "QCOM", "SWKS", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "XOM", "PG"],
    api_key=API_KEY,
    start_date="2019-01-01",
    end_date="2023-01-01",
)

**Does anything here explain Apple's returns at all?** An OLS regression of Apple's weekly returns on every other ticker at once controls for all regressors simultaneously -- unlike a raw pairwise correlation, which can mislead when regressors are correlated with each other. It reports the coefficient, Standard Error, t-Statistic and P-Value for each.

In [3]:
# AAPL is the Toolkit's first ticker, so it's the default dependent ticker;
# every other ticker becomes the default independent set
companies.econometrics.get_ols(period="weekly")

2026-08-04 12:55:36 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-08-04 12:55:37 - financetoolkit - INFO - Obtaining historical data for 12 ticker(s)
2026-08-04 12:55:40 - financetoolkit - INFO - Dependent ticker: AAPL | Independent ticker(s): ['TSM', 'QCOM', 'SWKS', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'XOM', 'PG']


,Coefficient,Std. Error,t-Statistic,P-Value
Intercept,0.002,0.0014,1.369,0.172
TSM,-0.003,0.0462,-0.0649,0.9483
QCOM,0.1237,0.0344,3.5982,0.0004
SWKS,0.2402,0.0425,5.6529,0.0
MSFT,0.2689,0.0722,3.7269,0.0002
GOOGL,0.132,0.0554,2.3828,0.0178
AMZN,0.0405,0.0461,0.8788,0.3802
META,-0.0004,0.0342,-0.0104,0.9917
NVDA,0.0306,0.0357,0.8582,0.3915
XOM,-0.0456,0.0342,-1.3345,0.183


Only some of the initially plausible relationships survive: Apple's RF/modem chip suppliers `QCOM` and `SWKS` and megacap peers `MSFT` and `GOOGL` come out statistically significant (P-Value < 0.05), while `AMZN`, `META`, `NVDA` and the unrelated `XOM` do not. Two results stand out as counter-intuitive: `PG` (consumer staples, meant as an unrelated contrast) is significant, while Apple's own foundry partner `TSM` is not -- a first hint that multicollinearity between correlated regressors can hide or manufacture relationships in a multiple regression. `SWKS` has the strongest relationship of the group, so that is what we dig into next.

**Can we trust those P-Values?** OLS's t-tests assume well-behaved residuals; non-normal returns would undermine that. The **Jarque-Bera test** checks this for Apple and its three most relevant chip names.

In [4]:
jarque_bera = companies.econometrics.get_jarque_bera_test(period="quarterly", within_period=False)

jarque_bera[["AAPL", "SWKS", "QCOM", "TSM"]]

,AAPL,SWKS,QCOM,TSM
Jarque-Bera Statistic,0.3422,1.856,1.9661,1.0564
P-Value,0.8427,0.3953,0.3742,0.5897


None of the four are close to rejecting normality (all P-Values well above 0.05), so there is no red flag undermining the OLS significance calls above.

**Is the `SWKS` relationship predictive, or just same-period comovement?** `SWKS` had the strongest coefficient above, but that only shows the two moved together *in the same week*. **Granger causality** asks whether one ticker's past values help predict the other's returns beyond its own history -- evidence of a lead-lag relationship, not same-period correlation. `get_granger_causality` computes every ordered pair at once, so we compute it once and select the `AAPL`/`SWKS` pair.

In [5]:
granger_causality = companies.econometrics.get_granger_causality(period="weekly")

granger_causality.loc[[("AAPL", "SWKS"), ("SWKS", "AAPL")]]

2026-08-04 12:55:40 - financetoolkit - INFO - Computing 110 dependent/independent ticker pairs


,,F-Statistic,P-Value,Granger-Causes (5%)
Dependent,Independent,,,
AAPL,SWKS,0.895078521304175,0.48466457570909804,False
SWKS,AAPL,0.944495404258346,0.45242719661032704,False


Neither direction is significant (P-Values of 0.47 and 0.31): the `SWKS`-`AAPL` relationship found above is contemporaneous -- they move together in the same week -- not a predictive, lead-lag relationship in either direction.

**What about the long run?** `TSM` is Apple's actual foundry partner, yet its returns weren't significant above. Short-run returns and long-run price levels answer different questions -- two series can share a long-run equilibrium (**cointegration**) without any short-run return relationship. First, the **Augmented Dickey-Fuller (ADF)** test confirms the price levels need this treatment: its null hypothesis is that a series is *non-stationary* (has a unit root), true of price levels but not returns.

In [6]:
adf = companies.econometrics.get_augmented_dickey_fuller(period="quarterly")

display(adf[["AAPL", "TSM"]])

cointegration = companies.econometrics.get_engle_granger_cointegration(period="quarterly")

display(cointegration.loc[[("AAPL", "TSM"), ("TSM", "AAPL")]])

,AAPL,TSM
ADF Statistic,-0.6802601805265639,-4.030001918273707
P-Value,0.8517965573320353,0.0012613087504334333
Lags Used,0,9
Observations,23,14
Critical Value 1%,-3.7529275211638033,-4.01203360058309
Critical Value 5%,-2.998499866852963,-3.1041838775510207
Critical Value 10%,-2.6389669754253307,-2.6909873469387753
Reject Unit Root (5%),False,True


2026-08-04 12:55:40 - financetoolkit - INFO - Computing 110 dependent/independent ticker pairs


,,Engle-Granger Statistic,P-Value,Critical Value 1%,Critical Value 5%,Critical Value 10%,Cointegrated (5%)
Dependent,Independent,,,,,,
AAPL,TSM,-1.0627051184389342,0.8902946614373253,-4.43598763705104,-3.6146844423440454,-3.233991776937618,False
TSM,AAPL,-2.3669102289269492,0.3406828476192678,-4.43598763705104,-3.6146844423440454,-3.233991776937618,False


Apple's price fails to reject the unit root (P-Value 0.53), as expected for a price level. `TSM`'s price is a more borderline case here given the short quarterly sample (only 11-15 observations), rejecting at the 5% level -- worth treating with some caution rather than at face value. Either way, the **Engle-Granger** test finds no cointegrating relationship between the two (P-Value 0.99): even Apple's own foundry partner shows no detectable long-run price equilibrium with Apple over this window, at least not with this few observations.

**Zoom out: does the whole panel share a common factor?** Rather than one pair at a time, `get_fixed_effects` treats every remaining ticker as an entity in a panel and asks whether they share a common sensitivity to `SWKS`, after removing each entity's own time-invariant average return.

In [7]:
fixed_effects = companies.econometrics.get_fixed_effects(
    independent_tickers="SWKS",
    dependent_tickers=[
        "TSM", "QCOM", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "XOM", "PG"
    ],
    period="weekly",
)

fixed_effects

,Coefficient,Std. Error,t-Statistic,P-Value
SWKS,0.4496,0.0165,27.1774,0.0
Entity Effect: AMZN,0.0032,NaN,NaN,NaN
Entity Effect: GOOGL,0.0031,NaN,NaN,NaN
Entity Effect: META,0.0029,NaN,NaN,NaN
Entity Effect: MSFT,0.0046,NaN,NaN,NaN
Entity Effect: NVDA,0.0085,NaN,NaN,NaN
Entity Effect: PG,0.0012,NaN,NaN,NaN
Entity Effect: QCOM,0.0033,NaN,NaN,NaN
Entity Effect: TSM,0.0031,NaN,NaN,NaN
Entity Effect: XOM,0.0008,NaN,NaN,NaN


The whole panel -- including `XOM` and `PG`, the two names picked specifically for being unrelated -- shares a highly significant sensitivity to `SWKS` (P-Value 0.0). That is the tell: since the Benchmark was deliberately excluded from this analysis, `SWKS`'s return here is standing in for general market-wide comovement rather than a genuine semiconductor-supply-chain effect. A cleaner version of this test would include the market factor explicitly and check what, if anything, is left over -- a good next step to try with `include_benchmark=True` on `get_ols` above.

Using the individual models with your own DataFrames is also a possibility thanks to the architecture of the Finance Toolkit.

In [8]:
import numpy as np

from financetoolkit.econometrics import diagnostics_model, regression_model

np.random.seed(42)

factor = pd.Series(np.random.normal(0, 1, 100), name="Factor")
asset_return = 2.5 * factor + pd.Series(np.random.normal(0, 0.5, 100))
asset_return.name = "Asset Return"

result = regression_model.get_ols(asset_return, factor)

display(regression_model.regression_summary_table(result))

display(diagnostics_model.get_jarque_bera_test(asset_return))

,Coefficient,Std. Error,t-Statistic,P-Value
Intercept,0.0037139149319834613,0.047790543502372745,0.07771233929990919,0.9382153254054465
Factor,2.4283714198642787,0.05254226681028179,46.21748484191931,2.3945876718316904e-68


Jarque-Bera Statistic   0.09214083237687773
P-Value                  0.9549747141436413
dtype: float64